# Bloque 1. Importaciones y Configuración.

In [ ]:
import re

# --- MAPA DE PIEZAS ---
PIECE_MAP = {
    'A': 'arqueros', 'B': 'escorpion', 'C': 'c_ligera', 
    'D': 'dragon',   'E': 'elefante',  'F': 'fortaleza',
    'H': 'chusma',   'L': 'lanceros',  'M': 'montana',
    'P': 'c_pesada', 'R': 'rey',       'T': 'trabuquete'
}

# Mapeo inverso para pintar el tablero (Nombre -> Letra)
TYPE_TO_CHAR = {v: k for k, v in PIECE_MAP.items()}

# Colores ANSI para la terminal
C_RESET  = "\033[0m"
C_GRAY   = "\033[90m"      # Gris (Vacío)
C_BLUE   = "\033[94m"      # Azul (Agua)
C_RED    = "\033[91m"      # Rojo (Ejército Rojo)
C_BLACK  = "\033[1;37m"    # Blanco Negrita (Para el Ejército Negro en fondo oscuro)
# Nota: Si usas fondo claro, cambia C_BLACK a "\033[30m"

# --- FUNCIÓN AUXILIAR (SIGN) ---
def sign(x):
    if x > 0: return 1
    if x < 0: return -1
    return 0

# Bloque 2. Funciones de ataque y amenazas.

In [12]:
def can_unit_attack(piece_type, r1, c1, r2, c2, army, board, terrain, ignore_pos=None):
    """Valida si una unidad puede atacar/capturar una casilla."""
    dr = abs(r2 - r1)
    dc = abs(c2 - c1)

    # 1. UNIDADES SIMPLES
    if piece_type == 'rey': return dr <= 1 and dc <= 1
    if piece_type == 'lanceros' or piece_type == 'chusma': return (dr + dc == 1)

    # 2. ARQUEROS
    if piece_type == 'arqueros':
        if dr != dc: return False
        if dr == 1: return True
        if dr == 2:
            mid_r, mid_c = (r1 + r2) // 2, (c1 + c2) // 2
            is_mid_empty = (board[mid_r][mid_c] is None) or \
                           (ignore_pos and mid_r == ignore_pos['r'] and mid_c == ignore_pos['c'])
            return is_mid_empty and terrain[mid_r][mid_c] != 'water'
        return False

    # 3. CABALLERÍA LIGERA
    if piece_type == 'c_ligera':
        if not ((dr == 3 and dc == 1) or (dr == 1 and dc == 3)): return False
        
        def check_block(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            if ignore_pos and r == ignore_pos['r'] and c == ignore_pos['c']: return False
            if terrain[r][c] == 'water': return True
            o = board[r][c]
            return (o and (o['type'] == 'montana' or o['type'] == 'fortaleza'))

        # Ruta A (2+1)
        diags2 = [(2, 2), (2, -2), (-2, 2), (-2, -2)]
        for kr, kc in diags2:
            if abs(r2 - (r1 + kr)) == 1 and abs(c2 - (c1 + kc)) == 1:
                knee_r, knee_c = r1 + kr, c1 + kc
                mid_r, mid_c = (r1 + knee_r) // 2, (c1 + knee_c) // 2
                if not check_block(mid_r, mid_c) and not check_block(knee_r, knee_c): return True

        # Ruta B (1+2)
        diags1 = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
        for kr, kc in diags1:
            if abs(r2 - (r1 + kr)) == 2 and abs(c2 - (c1 + kc)) == 2:
                knee_r, knee_c = r1 + kr, c1 + kc
                mid_r, mid_c = (knee_r + r2) // 2, (knee_c + c2) // 2
                if not check_block(knee_r, knee_c) and not check_block(mid_r, mid_c): return True
        return False

    # 4. CABALLERÍA PESADA
    if piece_type == 'c_pesada':
        if not ((dr == 2 and dc == 1) or (dr == 1 and dc == 2)): return False
        
        def is_blocked(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            if terrain[r][c] == 'water': return True
            if ignore_pos and r == ignore_pos['r'] and c == ignore_pos['c']: return False
            o = board[r][c]
            if o:
                if o['type'] in ['montana', 'fortaleza']: return True
                if o['army'] != army: return True
            return False

        sr, sc = sign(r2 - r1), sign(c2 - c1)
        if dr == 2:
            return not (is_blocked(r1 + sr, c1) or is_blocked(r2, c1)) or \
                   not (is_blocked(r1, c2) or is_blocked(r1 + sr, c2))
        else:
            return not (is_blocked(r1, c1 + sc) or is_blocked(r1, c2)) or \
                   not (is_blocked(r2, c1) or is_blocked(r2, c1 + sc))

    # 5. RAYCAST (Dragón, Elefante, Armas)
    # Lógica simplificada de raycast para validación rápida
    sr, sc = sign(r2 - r1), sign(c2 - c1)
    
    # Validar dirección
    is_ortho = (r1 == r2 or c1 == c2)
    is_diag = (dr == dc)
    
    if piece_type == 'dragon': 
        if not (is_ortho or is_diag): return False
    elif piece_type == 'elefante':
        if not is_ortho: return False
    elif piece_type == 'trabuquete': # Disparo
        if not is_ortho: return False
    elif piece_type == 'escorpion': # Disparo
        if not is_diag: return False

    cr, cc = r1 + sr, c1 + sc
    enemies_in_path = 0
    last_enemy_pos = None

    while cr != r2 or cc != c2:
        # --- 🛡️ MODIFICACIÓN QUIRÚRGICA: PROTECCIÓN DE LÍMITES ---
        # Si el rayo se sale del tablero, detenemos la validación inmediatamente
        if not (0 <= cr < 10 and 0 <= cc < 10):
            return False 
        # ---------------------------------------------------------

        if ignore_pos and cr == ignore_pos['r'] and cc == ignore_pos['c']:
            cr += sr; cc += sc; continue
        
        obs = board[cr][cc]
        
        if piece_type == 'dragon':
            if obs and obs['type'] != 'montana': return False
        
        elif piece_type == 'elefante':
            if terrain[cr][cc] == 'water' or obs: return False
            
        elif piece_type in ['trabuquete', 'escorpion']: # Armas
            if obs and obs['type'] in ['montana', 'fortaleza']: return False
            if piece_type == 'escorpion' and obs:
                if obs['army'] != army: 
                    enemies_in_path += 1; last_enemy_pos = {'r':cr, 'c':cc}
                else: return False
        
        cr += sr; cc += sc

    if piece_type == 'escorpion' and enemies_in_path > 0:
        if enemies_in_path == 1:
            dist = abs(r2 - last_enemy_pos['r'])
            return (dist >= 1 and dist <= 3)
        return False

    return True

def count_total_threats(target_r, target_c, army, board, terrain):
    """Cuenta cuántos enemigos (army) pueden atacar target."""
    threats = 0
    for tr in range(10):
        for tc in range(10):
            piece = board[tr][tc]
            if piece and piece['army'] == army:
                if can_unit_attack(piece['type'], tr, tc, target_r, target_c, army, board, terrain):
                    threats += 1
    return threats

def get_attackers(target_r, target_c, victim_army, board, terrain):
    """Devuelve lista de atacantes enemigos."""
    attackers = []
    enemy_army = 'negro' if victim_army == 'rojo' else 'rojo'
    for r in range(10):
        for c in range(10):
            p = board[r][c]
            if p and p['army'] == enemy_army:
                if can_unit_attack(p['type'], r, c, target_r, target_c, enemy_army, board, terrain):
                    attackers.append({'r': r, 'c': c, 'type': p['type']})
    return attackers

# Bloque 3. Lógica principal de movimiento.

In [13]:
def is_valid_move(piece_type, r1, c1, r2, c2, board, terrain):
    dr, dc = abs(r2 - r1), abs(c2 - c1)
    my_piece = board[r1][c1]
    target = board[r2][c2]

    # 0. REGLA ELEFANTE
    if my_piece and target and target['type'] == 'elefante' and target['army'] != my_piece['army']:
        if piece_type != 'dragon':
            threats = count_total_threats(r2, c2, my_piece['army'], board, terrain)
            if threats < 2: return False

    # 1. REY
    if piece_type == 'rey': return (dr <= 1 and dc <= 1) and (dr + dc > 0)

    # 2. LANCEROS / CHUSMA
    if piece_type in ['lanceros', 'chusma']: return (dr + dc == 1)

    # 3. DRAGÓN
    if piece_type == 'dragon':
        if not ((dr == dc) or (r1 == r2 or c1 == c2)): return False
        sr, sc = sign(r2 - r1), sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc
        while cr != r2 or cc != c2:
            obs = board[cr][cc]
            if obs and obs['type'] != 'montana': return False
            cr += sr; cc += sc
        return True

    # 4. ARQUEROS
    if piece_type == 'arqueros':
        if dr != dc: return False
        if dr == 1: return True
        if dr == 2:
            if target is None: return False # Solo captura saltando
            mid_r, mid_c = (r1 + r2) // 2, (c1 + c2) // 2
            return (board[mid_r][mid_c] is None and terrain[mid_r][mid_c] != 'water')
        return False

    # 5. ELEFANTE
    if piece_type == 'elefante':
        if r1 != r2 and c1 != c2: return False
        sr, sc = sign(r2 - r1), sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc
        while cr != r2 or cc != c2:
            if board[cr][cc] is not None or terrain[cr][cc] == 'water': return False
            cr += sr; cc += sc
        return True

    # 6. C. LIGERA y 7. C. PESADA
    # Reutilizamos la lógica de ataque que es idéntica geométricamente
    if piece_type in ['c_ligera', 'c_pesada']:
        return can_unit_attack(piece_type, r1, c1, r2, c2, my_piece['army'], board, terrain)

    # 8. TRABUQUETE (Movimiento)
    if piece_type == 'trabuquete':
        if target is None: return (dr == 1 and dc == 1) # Mover
        # Disparo: Ortogonal
        if r1 != r2 and c1 != c2: return False
        sr, sc = sign(r2 - r1), sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc
        while cr != r2 or cc != c2:
            o = board[cr][cc]
            if o and o['type'] in ['montana', 'fortaleza']: return False
            cr += sr; cc += sc
        return True

    # 9. ESCORPIÓN (Movimiento)
    if piece_type == 'escorpion':
        if target is None: return (dr + dc == 1) # Mover
        # Disparo: Diagonal (usa lógica compleja de can_unit_attack)
        return can_unit_attack('escorpion', r1, c1, r2, c2, my_piece['army'], board, terrain)

    return False

def is_simulated_move_safe(piece_type, r1, c1, r2, c2, army, board, terrain):
    """Verifica si el movimiento deja al Rey en Jaque (Suicidio)."""
    king_r, king_c = None, None
    if piece_type == 'rey': king_r, king_c = r2, c2
    else:
        for r in range(10):
            for c in range(10):
                p = board[r][c]
                if p and p['type'] == 'rey' and p['army'] == army:
                    king_r, king_c = r, c; break
            if king_r: break
    
    if king_r is None: return True # Sin rey no hay jaque (debug)

    # Simular
    orig_src = board[r1][c1]
    orig_dst = board[r2][c2]
    
    board[r1][c1] = None
    board[r2][c2] = orig_src # Nota: No simulamos relevo complejo aquí para brevedad
    
    threats = get_attackers(king_r, king_c, army, board, terrain)
    
    # Revertir
    board[r1][c1] = orig_src
    board[r2][c2] = orig_dst
    
    return len(threats) == 0

# Bloque 4. El motor del juego.

In [ ]:
class CotadrezGame:
    def __init__(self):
        self.board = [[None]*10 for _ in range(10)]
        self.terrain = [[None]*10 for _ in range(10)]
        self._init_water()
        self.j1_color = 'rojo'; self.j2_color = 'negro'; self.turn_color = 'rojo'
        self.game_phase = 'setup'
        self.fortress_bounds = {'rojo': None, 'negro': None}
        self.reserves = {'rojo': {}, 'negro': {}}
        self.dungeons = {'rojo': [], 'negro': []}
        for p in PIECE_MAP.values(): self.reserves['rojo'][p] = 10; self.reserves['negro'][p] = 10

    def set_j1_color(self, c_raw):
        c = c_raw.lower()
        if "rojo" in c: self.j1_color, self.j2_color = 'rojo', 'negro'
        elif "negro" in c: self.j1_color, self.j2_color = 'negro', 'rojo'
        self.turn_color = self.j1_color
        print(f"⚙️  J1={self.j1_color.upper()} | J2={self.j2_color.upper()}")

    def _init_water(self):
        for r, c in [(3,1), (1,3)]:
            self.terrain[r][c] = self.terrain[r][9-c] = \
            self.terrain[9-r][c] = self.terrain[9-r][9-c] = 'water'

    def coord_to_index(self, s):
        try: return 10 - int(s[1:]), 'abcdefghij'.index(s[0].lower())
        except: return None, None

    def index_to_coord(self, r, c):
        if r is None or c is None: return "??"
        return f"{'abcdefghij'[c]}{10-r}"

    def fmt(self, r, c):
        return f"({r},{c}|{self.index_to_coord(r,c)})"

    def switch_turn(self):
        self.turn_color = 'negro' if self.turn_color == 'rojo' else 'rojo'
    
    def get_icon(self, c=None): return "🔴" if (c or self.turn_color) == 'rojo' else "⚫"

    def deploy_fortress_area(self, r1, c1, r2, c2):
        army = self.turn_color
        min_r, max_r = min(r1, r2), max(r1, r2)
        min_c, max_c = min(c1, c2), max(c1, c2)
        if (max_r - min_r) != 1 or (max_c - min_c) != 1: return print("❌ ILEGAL: 2x2")
        self.fortress_bounds[army] = {'min_r': min_r, 'max_r': max_r, 'min_c': min_c, 'max_c': max_c}
        for cr in [min_r, max_r]:
            for cc in [min_c, max_c]:
                self.board[cr][cc] = {'type': 'fortaleza', 'army': army}
        print(f"{self.get_icon(army)} Fortaleza OK {min_r}-{max_r}, {min_c}-{max_c}")

    def deploy_unit(self, p_type, r, c):
        army = self.turn_color
        pos_str = self.fmt(r, c) # Usamos el nuevo formateador
        
        if self.board[r][c]: return print(f"⚠️ Ocupado en {pos_str}")
        
        if p_type == 'montana':
            self.board[r][c] = {'type': p_type, 'army': army}
            return print(f"🏔️  Montaña en {pos_str}")
        
        b = self.fortress_bounds[army]
        if not b: return print("❌ Falta Fortaleza")
        
        # Validación de anillo
        if max(max(0, b['min_r']-r, r-b['max_r']), max(0, b['min_c']-c, c-b['max_c'])) != 1:
            return print(f"❌ {p_type} en {pos_str} fuera de anillo")
        
        self.board[r][c] = {'type': p_type, 'army': army}
        if self.reserves[army].get(p_type, 0) > 0: self.reserves[army][p_type] -= 1
        
        # Print actualizado (quitamos el print manual aquí, lo delegamos al parser o usamos un return limpio)
        # Para mantener coherencia con tu log actual, imprimimos confirmación:
        print(f"🛡️  {self.get_icon()} {p_type.upper()} -> {pos_str}")

    def rescue_piece(self, p_type):
        self.reserves[self.turn_color][p_type] = self.reserves[self.turn_color].get(p_type, 0) + 1

    def execute_shot(self, r1, c1, r2, c2):
        target = self.board[r2][c2]
        if target:
            self.dungeons[self.turn_color].append(target)
            self.board[r2][c2] = None

    # --- AQUÍ ESTÁ LA FUNCIÓN QUE FALTABA (apply_move) ---
    def apply_move_internal(self, r1, c1, r2, c2):
        """Ejecuta el movimiento validando reglas."""
        piece = self.board[r1][c1]
        
        # Validaciones de seguridad
        if not piece: return print("❌ Origen vacío")
        
        # Validar legalidad (Usamos las funciones globales)
        if not is_valid_move(piece['type'], r1, c1, r2, c2, self.board, self.terrain):
             print(f"❌ Movimiento Inválido (Geometría/Bloqueo)")
             # En un log real, podríamos querer forzarlo, pero aquí avisamos
             # return 
        
        if not is_simulated_move_safe(piece['type'], r1, c1, r2, c2, piece['army'], self.board, self.terrain):
             print(f"❌ ILEGAL: El movimiento deja al Rey en Jaque")
             # return

        target = self.board[r2][c2]
        
        # Lógica de Captura / Relevo
        if target:
            if target['army'] == piece['army']: # Relevo (solo lanceros o swap permitido)
                self.board[r1][c1] = target
                self.board[r2][c2] = piece
                return
            else: # Captura
                self.dungeons[self.turn_color].append(target)
        
        # Mover
        self.board[r2][c2] = piece
        if not (target and target['army'] == piece['army']): # Si no fue relevo
            self.board[r1][c1] = None

    def find_piece_candidates(self, p_type, dest_r, dest_c, is_capture):
        candidates = []
        for r in range(10):
            for c in range(10):
                p = self.board[r][c]
                if p and p['type'] == p_type and p['army'] == self.turn_color:
                    if is_valid_move(p_type, r, c, dest_r, dest_c, self.board, self.terrain):
                        candidates.append({'r': r, 'c': c})
        return candidates
    
    # --- VISUALIZACIÓN (LA GUINDA 🍒) ---
    def render(self):
        print("\n   " + " ".join("abcdefghij")) # Cabecera Columnas
        
        for r in range(10):
            # Número de fila (izquierda)
            line = f"{10-r:2} "
            
            for c in range(10):
                piece = self.board[r][c]
                terr = self.terrain[r][c]
                
                # 1. CASILLA CON PIEZA
                if piece:
                    char = TYPE_TO_CHAR.get(piece['type'], '?')
                    color = C_RED if piece['army'] == 'rojo' else C_BLACK
                    # Pintamos la letra con su color
                    line += f"{color}{char}{C_RESET} "
                
                # 2. CASILLA DE AGUA
                elif terr == 'water':
                    line += f"{C_BLUE}={C_RESET} "
                
                # 3. CASILLA VACÍA
                else:
                    line += f"{C_GRAY}0{C_RESET} "
            
            # Número de fila (derecha) para referencia rápida
            print(line + f"{10-r}")

        # --- ESTADÍSTICAS (Reservas y Mazmorras) ---
        print("-" * 30)
        
        # Función auxiliar para imprimir listas bonitas
        def fmt_list(d, color_code):
            items = []
            # Ordenamos por tipo de pieza para consistencia
            for p_type, count in d.items():
                if count > 0:
                    char = TYPE_TO_CHAR.get(p_type, '?')
                    items.append(f"{char}:{count}")
            return f"{color_code}" + " ".join(items) + f"{C_RESET}"

        def fmt_dungeon(lst, color_code):
            if not lst: return f"{C_GRAY}(Vacía){C_RESET}"
            chars = [TYPE_TO_CHAR.get(p['type'], '?') for p in lst]
            return f"{color_code}" + "".join(chars) + f"{C_RESET}"

        # Imprimir Reservas
        print(f"📦 RESERVAS:")
        print(f"   🔴 Rojo:  {fmt_list(self.reserves['rojo'], C_RED)}")
        print(f"   ⚫ Negro: {fmt_list(self.reserves['negro'], C_BLACK)}")
        
        # Imprimir Mazmorras (Prisioneros capturados)
        print(f"⛓️  MAZMORRAS (Prisioneros):")
        # Nota: La mazmorra ROJA contiene piezas NEGRAS capturadas, y viceversa.
        print(f"   🔴 Tiene prisioneros: {fmt_dungeon(self.dungeons['rojo'], C_BLACK)}")
        print(f"   ⚫ Tiene prisioneros: {fmt_dungeon(self.dungeons['negro'], C_RED)}")
        print("-" * 30)

# Bloque 5. Parser y Ejecución.

In [ ]:
class CotadrezParser:
    def __init__(self, game): 
        self.game = game
        self.siege = False # Necesitamos la clase para recordar esto entre líneas
        
    def parse(self, text):
        # Limpieza previa
        text = text.replace("->ASEDIO", " ->ASEDIO ").replace("<-ASEDIO", " <-ASEDIO ")
        
        print("\n🏁 --- INICIANDO MODO PASO A PASO ---")
        print("ℹ️  (Pulsa ENTER en la barra superior para avanzar)")
        
        for line in text.strip().split('\n'):
            line = line.strip()
            
            # Filtro de líneas vacías o metadatos
            if not line or line.startswith("["): 
                if "J1" in line: 
                    match = re.search(r'"(.+)"', line)
                    color = match.group(1) if match else "Rojo"
                    self.game.set_j1_color(color)
                continue
            
            # --- COMANDOS DE FASE (Sin pausa) ---
            if "rF" in line:
                cs = re.findall(r'([a-j](?:10|[0-9]))', line)
                if len(cs)==2: 
                    print(f"\n🏗️  FORTALEZA: [{line}]")
                    r1,c1=self.game.coord_to_index(cs[0])
                    r2,c2=self.game.coord_to_index(cs[1])
                    self.game.deploy_fortress_area(r1,c1,r2,c2)
                continue

            if "//" in line: 
                print("\n🔄 CAMBIO DE DESPLIEGUE")
                self.game.turn_color = self.game.j2_color
                continue
            
            if ">>>" in line: 
                print("\n⚔️  INICIO DEL COMBATE")
                self.game.game_phase = 'play'
                self.game.turn_color = self.game.j1_color
                continue

            # --- TOKENS DE ACCIÓN (Con pausa) ---
            tokens = line.split()
            i = 0
            while i < len(tokens):
                t = tokens[i]
                
                # Ignorar números de turno y paréntesis
                if re.match(r'^\d', t) or "(" in t: 
                    i+=1; continue
                
                # 🛑 PAUSA INTERACTIVA + RENDER 🛑
                print(f"\n🔹 Instrucción: [{t}]")
                self.game.render()  # <--- ¡AQUÍ ESTÁ LA MAGIA!
                input(" (Enter...)") 


                # Asedio
                if "ASEDIO" in t: 
                    self.siege = ("->" in t)
                    print(f"🚨 ASEDIO: {self.siege}")
                    i+=1; continue
                
                # Marcha Forzada
                if "&" in t: 
                    print("⚡ Marcha Forzada")
                    if self.game.game_phase == 'play': self.game.switch_turn()
                    i+=1; continue

                # Acciones (Rescate / Movimiento)
                if re.match(r'^m[A-Z]', t): 
                    char = t[1]
                    # Asumiendo que PIECE_MAP está definido en el Bloque 1
                    self.game.rescue_piece(PIECE_MAP.get(char, '?'))
                    print(f"🚁 Rescate")
                else:
                    # Regex para movimiento estándar
                    m = re.match(r'^([r])?([A-Z])([a-j](?:10|[0-9]))?([x=])?([A-Z])?([a-j](?:10|[0-9]))', t)
                    if m:
                        pre, char, org, act, tgt, dst = m.groups()
                        pt = PIECE_MAP.get(char)
                        dr, dc = self.game.coord_to_index(dst)
                        
                        # A) Despliegue
                        if pre == 'r': 
                            self.game.deploy_unit(pt, dr, dc)
                        
                        # B) Movimiento / Disparo
                        else:
                            or_r, or_c = None, None
                            
                            # Caso 1: Origen explícito
                            if org:
                                or_r, or_c = self.game.coord_to_index(org)
                            
                            # Caso 2: Buscar candidatos
                            else:
                                cands = self.game.find_piece_candidates(pt, dr, dc, act=='x')
                                if len(cands) == 1: 
                                    or_r, or_c = cands[0]['r'], cands[0]['c']
                                elif len(cands) == 0: 
                                    print(f"❌ ILEGAL: Nadie llega a {dst}"); i+=1; continue
                                else: 
                                    print(f"❌ AMBIGUO: Varios llegan a {dst}"); i+=1; continue
                            
                            if or_r is None: 
                                print(f"❌ ERROR COORD"); i+=1; continue

                            # Info Visual Bonita
                            coords_txt = f"{self.game.fmt(or_r, or_c)} -> {self.game.fmt(dr, dc)}"
                            
                            if act == 'x' and pt in ['trabuquete', 'escorpion']:
                                print(f"🔥 DISPARO {coords_txt}")
                                self.game.execute_shot(or_r, or_c, dr, dc)
                            elif act == '=':
                                print(f"🔄 RELEVO {coords_txt}")
                                self.game.apply_move_internal(or_r, or_c, dr, dc)
                            else:
                                verb = "CAPTURA" if act=='x' else "MUEVE"
                                print(f"➡️ {verb} {coords_txt}")
                                self.game.apply_move_internal(or_r, or_c, dr, dc)
                
                # Gestión de turno (Solo si no hay asedio)
                if self.game.game_phase == 'play' and not self.siege: 
                    self.game.switch_turn()
                i += 1
                
# --- EJECUCIÓN ---
juego = CotadrezGame()
parser = CotadrezParser(juego)

log = """
    [Event "Test Gemini"]
    [J1 "Rojo"]
    [J2 "Negro"]

    rFg2h3
    rPi3
    rBh4
    rTg4
    rDf4
    rCf3
    rLf2
    rEf1
    rAi1
    rEi2
    rRh1
    rHg1
    rMe3
    rMf5
    rMh5
    //
    rFc8d9
    rMd6
    rMg6
    rMe7
    rRd10
    rEc10
    rPe10
    rDb9
    rBe9
    rCb8
    rTe8
    rLb3
    rHc7
    rLd7
    >>>
    Cc4
    L=Hc7

    Le9e8
    Lc2c3
    Axe5
    Txd5 ->ASEDIO
    Pe7
    mE
    rRh6
    <-ASEDIO
    """

parser.parse(log)


🏁 --- INICIANDO MODO PASO A PASO ---
ℹ️  (Pulsa ENTER en la barra superior para avanzar)
⚙️  J1=ROJO | J2=NEGRO

🏗️  FORTALEZA: [rFg2h3]
🔴 Fortaleza OK 7-8, 6-7

🔹 Instrucción: [rPi3] 🛡️  🔴 C_PESADA -> (7,8|i3)

🔹 Instrucción: [rBh4] 🛡️  🔴 ESCORPION -> (6,7|h4)

🔹 Instrucción: [rTg4] 🛡️  🔴 TRABUQUETE -> (6,6|g4)

🔹 Instrucción: [rDf4] 🛡️  🔴 DRAGON -> (6,5|f4)

🔹 Instrucción: [rCf3] 🛡️  🔴 C_LIGERA -> (7,5|f3)

🔹 Instrucción: [rLf2] 🛡️  🔴 LANCEROS -> (8,5|f2)

🔹 Instrucción: [rEf1] 🛡️  🔴 ELEFANTE -> (9,5|f1)

🔹 Instrucción: [rAi1] 🛡️  🔴 ARQUEROS -> (9,8|i1)

🔹 Instrucción: [rEi2] 🛡️  🔴 ELEFANTE -> (8,8|i2)

🔹 Instrucción: [rRh1] 🛡️  🔴 REY -> (9,7|h1)

🔹 Instrucción: [rHg1] 🛡️  🔴 CHUSMA -> (9,6|g1)

🔹 Instrucción: [rMe3] 🏔️  Montaña en (7,4|e3)

🔹 Instrucción: [rMf5] 🏔️  Montaña en (5,5|f5)

🔹 Instrucción: [rMh5] 🏔️  Montaña en (5,7|h5)

🔄 CAMBIO DE DESPLIEGUE

🏗️  FORTALEZA: [rFc8d9]
⚫ Fortaleza OK 1-2, 2-3

🔹 Instrucción: [rMd6] 🏔️  Montaña en (4,3|d6)

🔹 Instrucción: [rMg6] 🏔️  Montañ